# Fake News Detection — Applied to New Pre-Vectorized Dataset

This notebook re-runs the LR / NB / SVM / RF / XGBoost pipeline from
`ML_aniket_final.ipynb` against the **new dataset** you uploaded:

- `tfidf_sparse_matrix.npz` — 39,100 x 30,000 pre-built TF-IDF matrix
- `tfidf_labels.csv` — binary `label` column
- `tfidf_feature_names.npy` — 30,000 feature names (`title_tfidf__*`, `content_tfidf__*`)

**Important difference from the original notebook:** the original notebook
started from *raw text* (`cleaned_news_data.csv` with `title`/`content`/`subject`
columns) and did its own TF-IDF vectorization after three leakage fixes
(dropping `subject`, stripping the Reuters wire-service dateline, and
deduplicating on the raw text). Your new upload is **already vectorized** —
there is no raw text and no `subject` column in these files, so those
text-level checks can't be re-run directly on this data.

The good news: the feature name prefixes are only `title_tfidf__` /
`content_tfidf__` (no `subject_tfidf__`), and there is no `reuters` token in
the vocabulary at all — both are exactly what you'd expect if this matrix was
already built *after* the subject and dateline leakage fixes. So the two
biggest leaks from the original notebook look like they're already closed in
this dataset. The one check that **is** still possible directly on the
vectors is duplicate-row detection (a proxy for the duplicate-article leak),
which I ran below.

## 0. Imports & config

In [ ]:
import time, pickle, json, hashlib
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
    log_loss, matthews_corrcoef, cohen_kappa_score,
)

RANDOM_STATE = 42

## 1. Load the new dataset

In [ ]:
X = sp.load_npz('tfidf_sparse_matrix.npz').tocsr()
y = pd.read_csv('tfidf_labels.csv')['label'].values
feature_names = np.load('tfidf_feature_names.npy', allow_pickle=True)

print(f"X: {X.shape}, y: {y.shape}")
print(pd.Series(y).value_counts(normalize=True).rename('class balance'))

## 2. Leakage check — duplicate TF-IDF vectors

Without raw text, exact/near-duplicate *articles* can't be checked directly,
but exact-duplicate **vectors** are a reasonable proxy: two rows with an
identical TF-IDF fingerprint are extremely likely to be the same (or
re-syndicated) article. Found **191 duplicate rows (0.49%)** — much lower
than the ~12.6% in the original raw-text dataset, again consistent with this
matrix already being past the dedup step. Dropped them before splitting
anyway, since any duplicate leaking across train/test still inflates
metrics.

In [ ]:
def row_hash(i):
    s, e = X.indptr[i], X.indptr[i+1]
    idx = X.indices[s:e].tobytes()
    dat = np.round(X.data[s:e], 6).tobytes()
    return hashlib.md5(idx + dat).hexdigest()

hashes = pd.Series([row_hash(i) for i in range(X.shape[0])])
keep_mask = (~hashes.duplicated(keep='first')).values
print(f"Dropping {(~keep_mask).sum()} duplicate-vector rows ({(~keep_mask).mean():.2%})")

X = X[keep_mask]
y = y[keep_mask]
print(f"After dedup: {X.shape}")

## 3. Train/test split (80/20, stratified)

In [ ]:
idx_train, idx_test, y_train, y_test = train_test_split(
    np.arange(X.shape[0]), y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_test = X[idx_train], X[idx_test]
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

**Caveat worth flagging:** because this matrix arrived pre-built, we
can't confirm the vectorizer was fit on *only* the training text before this
split — if whoever built the matrix fit TF-IDF on the full corpus first, the
IDF weights technically "saw" the test set's vocabulary distribution, which
is a mild leak we can't undo from vectors alone. If you still have the raw
text (`title`/`content` columns) or the fitted `TfidfVectorizer` objects,
send those over and I'll rebuild the matrix split-first to close this
gap completely.

## 4. Shared evaluation helper

In [ ]:
all_metrics = {}
all_models = {}

def evaluate(name, model, y_pred, y_proba, model_out=None, extra=None):
    m = {}
    m["accuracy"] = accuracy_score(y_test, y_pred)
    m["precision"] = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    m["recall"] = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    m["f1_score"] = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    m["roc_auc"] = roc_auc_score(y_test, y_proba)
    m["average_precision"] = average_precision_score(y_test, y_proba)
    m["log_loss"] = log_loss(y_test, y_proba)
    m["matthews_corrcoef"] = matthews_corrcoef(y_test, y_pred)
    m["cohen_kappa"] = cohen_kappa_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred)
    m["confusion_matrix"] = cm.tolist()
    m["classification_report"] = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    prec_c, rec_c, _ = precision_recall_curve(y_test, y_proba)
    m["roc_curve"] = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
    m["pr_curve"] = {"precision": prec_c.tolist(), "recall": rec_c.tolist()}

    if extra:
        m.update(extra)

    print("=" * 60)
    print(f"{name} - EVALUATION METRICS")
    print("=" * 60)
    for k, v in m.items():
        if isinstance(v, float):
            print(f"{k:>22}: {v:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    all_metrics[name] = m
    all_models[name] = model
    if model_out:
        with open(model_out, "wb") as f:
            pickle.dump({"model": model, "feature_names": feature_names}, f)
    return m

# Logistic Regression

In [ ]:
t0 = time.time()
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")

y_pred_lr = lr_model.predict(X_test)
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]
lr_metrics = evaluate("LR", lr_model, y_pred_lr, y_proba_lr, "logistic_regression_model.pkl")

# Naive Bayes

In [ ]:
t0 = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")

y_pred_nb = nb_model.predict(X_test)
y_proba_nb = nb_model.predict_proba(X_test)[:, 1]
nb_metrics = evaluate("NB", nb_model, y_pred_nb, y_proba_nb, "naive_bayes_model.pkl")

# SVM

`LinearSVC` wrapped in `CalibratedClassifierCV` (cv=5) for probability estimates.

In [ ]:
t0 = time.time()
base_svm = LinearSVC(max_iter=5000, dual="auto", random_state=RANDOM_STATE)
svm_model = CalibratedClassifierCV(base_svm, cv=5)
svm_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")

y_pred_svm = svm_model.predict(X_test)
y_proba_svm = svm_model.predict_proba(X_test)[:, 1]
svm_metrics = evaluate("SVM", svm_model, y_pred_svm, y_proba_svm, "svm_model.pkl")

# Random Forest

In [ ]:
t0 = time.time()
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=30, max_features="sqrt",
    n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced",
)
rf_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

importances = rf_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:20]
print("Top 20 most important features:")
for i in top_idx:
    print(f"  {feature_names[i]:<40} {importances[i]:.4f}")

rf_metrics = evaluate(
    "RF", rf_model, y_pred_rf, y_proba_rf, "random_forest_model.pkl",
    extra={"feature_importances": dict(zip(feature_names, importances.tolist()))},
)

# XGBoost

In [ ]:
t0 = time.time()
n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())
scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
    eval_metric="logloss", scale_pos_weight=scale_pos_weight,
)
xgb_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

importances = xgb_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:20]
print("Top 20 most important features:")
for i in top_idx:
    print(f"  {feature_names[i]:<40} {importances[i]:.4f}")

xgb_metrics = evaluate(
    "XGB", xgb_model, y_pred_xgb, y_proba_xgb, "xgboost_model.pkl",
    extra={"feature_importances": dict(zip(feature_names, importances.tolist()))},
)

## Final comparison — all five models

In [ ]:
print("=" * 70)
print("LR vs NAIVE BAYES vs SVM vs RANDOM FOREST vs XGBOOST")
print("=" * 70)
for k in ["accuracy", "precision", "recall", "f1_score", "roc_auc"]:
    row = f"{k:>12} |"
    for name in ["LR", "NB", "SVM", "RF", "XGB"]:
        row += f" {name}: {all_metrics[name][k]:.4f} |"
    print(row)

with open("all_evaluation_metrics.pkl", "wb") as f:
    pickle.dump(all_metrics, f)

## Results on your new dataset

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| LR | 0.9859 | 0.9859 | 0.9859 | 0.9859 | 0.9986 |
| NB | 0.9445 | 0.9447 | 0.9445 | 0.9445 | 0.9874 |
| SVM | 0.9891 | 0.9891 | 0.9891 | 0.9891 | 0.9991 |
| RF | 0.9720 | 0.9724 | 0.9720 | 0.9719 | 0.9968 |
| XGB | 0.9865 | 0.9865 | 0.9865 | 0.9865 | 0.9988 |

**Best overall: SVM** (98.91% accuracy, 0.999 ROC-AUC), with XGBoost and
Logistic Regression close behind (~98.6%). Naive Bayes is the clear
laggard (94.45%) — expected, since its independence assumption is the
weakest fit for TF-IDF features with this much correlation between tokens.

**Why these are still high (~94-99%) and that's believable, not a leak:**
the vocabulary and duplicate checks above indicate the subject/dateline/
duplicate-article leaks from the original raw-text notebook are already
absent here. What's left is a real, if unusually easy, property of this
corpus: the two classes map closely to two *writing styles/sources* (formal
wire-service style vs. informal/partisan style), which bag-of-words TF-IDF
picks up very easily. That's the same conclusion the original notebook
reached — it's not a new artifact introduced by this dataset.

**DistilBERT was skipped** — the transformer stage needs raw article text
to tokenize, and this upload only contains pre-built TF-IDF vectors (no
`title`/`content` strings). If you upload the raw text file used to build
this matrix, I can run that stage too.

**If you want to push accuracy/F1/ROC-AUC further from here:**
- Rebuild the TF-IDF matrix split-first (fit only on train text) if you have the raw text, to close the residual "IDF saw the test set" gap noted in Section 3.
- Hyperparameter-tune SVM's `C` and XGBoost's `max_depth`/`learning_rate`/`n_estimators` via grid or random search with cross-validation — the current runs use the same fixed hyperparameters as the original notebook.
- Try a soft-voting ensemble of SVM + XGB + LR (the three strongest here); ensembles of well-correlated-but-not-identical models often edge out the best single model by 0.1-0.5 points.
- Add char n-gram TF-IDF alongside the current word n-grams — catches misspellings/stylistic quirks the word-level vocabulary misses.
